# Records from PhysioNet

PhysioNet publishes its databases at
`https://physionet.org/files/<database>/<version>/`. wfdb-x reads a record
directly from there when you choose to: set that address as the **input
directory** and allow **network access**. Both are explicit choices: by
default, wfdb-x reads local files only, never contacts a server on its own,
and never searches for a record in several places (the original WFDB
library could silently fall back to physionet.org, or open a stale copy of a
record found earlier in its search path).

This notebook reads record 100 of the MIT-BIH Arrhythmia Database (`mitdb`,
version 1.0.0; about 2 MB). Without a network connection, it uses the sample
record `100s` shipped with wfdb-x (the first minute of the same record)
instead.

In [ ]:
import numpy as np
import wfdbx

PHYSIONET = "https://physionet.org/files/mitdb/1.0.0/"

# The explicit choice: read under the PhysioNet address, with network access
wfdbx.wfdbx_set_input_dir(PHYSIONET)
wfdbx.wfdbx_set_network(1)
try:
    hdr = wfdbx.read_header("100")        # reads PHYSIONET + "100.hea"
    record = "100"
except IOError:
    print("Could not read record 100 from PhysioNet (no network connection?);")
    print("using the sample record 100s shipped with wfdb-x instead.")
    wfdbx.wfdbx_set_network(0)
    wfdbx.wfdbx_set_input_dir(wfdbx.sample_data_dir())
    hdr = wfdbx.read_header("100s")
    record = "100s"
print("input directory:", wfdbx.wfdbx_get_input_dir())

In [ ]:
rec = wfdbx.read_record(record, sampto="0:10", physical=True)
ann = wfdbx.read_annotations(record, "atr")
print(f"record {record}: {hdr.nsig} signals at {rec.fs} Hz, "
      f"{len(ann)} annotations")
print(rec.names, rec.signals[:3])

Each remote file is read once per session, in full, when it is first needed:
the header, then the signal file it names (in the same directory as the
header), then the annotation file. Every relative record name is read under
the input directory, so `read_header("100")` reads
`https://physionet.org/files/mitdb/1.0.0/100.hea`; nothing else is tried.

The same for a single call, without changing the settings:

```python
rec = wfdbx.read_record("100", input_dir="https://physionet.org/files/mitdb/1.0.0/",
                        network=True)
```

and with the command-line tools, through environment variables:

```
WFDBX_NETWORK=1 WFDBX_INPUT_DIR=https://physionet.org/files/mitdb/1.0.0/ wfdbx rdsamp -r 100 -t 0:5
```

The same works for any PhysioNet database: use its `files` address
(`https://physionet.org/files/<database>/<version>/`); its `RECORDS` file lists
the names of its records. To work offline, download the files of a record
(its header, the signal files named in the header, and its annotation files)
into a directory and use that directory as the input directory.

In [ ]:
# Back to the defaults: local files only, under the current directory
wfdbx.wfdbx_set_input_dir(None)
wfdbx.wfdbx_set_network(0)
print("network access:", wfdbx.wfdbx_get_network(),
      "- input directory:", wfdbx.wfdbx_get_input_dir())